# Building an informed chatbot

In this lab you'll be using pretrained LLM models to create a chatbot that can answer questions about a document like MIDREGS.

[Here's your Gem](https://gemini.google.com/gem/1fuedJfxB0OZ-ykvjKzHjPY5Z4fkUhPTr?usp=sharing)

## Step 0: Create your environment

- `mamba create -n llm_rag transformers huggingface_hub numpy scikit-learn https requests pypdf ipykernel jupyter -y`
- `mamba activate llm_rag`
- `pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu130`

## Step 1: Create source text file

We need to make a `.txt` file which contains the text we want to have as a resource. [MIDREGS](https://www.usna.edu/Commandant/_files/COMDTMIDNINST_5400.7_MIDSHIPMEN_REGULATIONS_MANUAL.pdf) is a good place to start, though ultimately you're welcome to use your own source documents. You can either copy-paste the text into a `.txt` file or use the below program to extract it for you:

```python
import sys
from pypdf import PdfReader

def extract_text(filename):
    try:
        reader = PdfReader(filename)
        for page in reader.pages:
            text = page.extract_text()
            if text:
                sys.stdout.write(text + "\n")
    except FileNotFoundError:
        sys.stderr.write(f"Error: The file '{filename}' was not found.\n")
        sys.exit(1)
    except Exception as e:
        sys.stderr.write(f"An error occurred: {e}\n")
        sys.exit(1)

if __name__ == "__main__":
    if len(sys.argv) != 2:
        sys.stderr.write("Usage: python extract_pdf.py <filename.pdf>\n")
        sys.exit(1)
    
    target_file = sys.argv[1]
    extract_text(target_file)
```

## Step 2: Understand the provided functions

This first cell downloads our two models. We are using separate models for generation and embeddings. Models for generation ("causal" models) are intended to understand everything coming before a token, but not expecting anything after - this makes them good for generating text, but not for generating embeddings of full sentences. Models for embedding are built to consider the full sentence. We are using `Qwen2.5-3B-Instruct` as our causal model, and `all-MiniLM-L6-v2` as our embedding model. Both are modern but small. We are downloading them from Huggingface.

We define a few variables here:
- `tokenizer`: tokenizes text as expected by `Qwen`.
- `model`: the `Qwen` causal model.
- `embed_tokenizer`: tokenizes text as expected by `all-MiniLM-L6-v2`.
- `embed_model`: the `all-MiniLM-L6-v2` text embedding model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel

import os, re
import httpx
from huggingface_hub import constants
import numpy as np
from sklearn.neighbors import NearestNeighbors

# 1. Standard environment bypasses
os.environ['HF_HUB_DISABLE_SSL_VERIFY'] = '1'
os.environ['CURL_CA_BUNDLE'] = ''

# 2. Force httpx to ignore SSL structural errors (Missing Authority Key Identifier)
# We wrap the original client to ensure verify=False is always passed.
original_client = httpx.Client

class UnverifiedClient(original_client):
    def __init__(self, *args, **kwargs):
        kwargs['verify'] = False
        super().__init__(*args, **kwargs)

httpx.Client = UnverifiedClient

# 3. Suppress warnings
import warnings
from urllib3.exceptions import InsecureRequestWarning
import requests
requests.packages.urllib3.disable_warnings(category=InsecureRequestWarning)
warnings.filterwarnings('ignore')

# Load tokenizer and causal model
model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to('cuda')
model.eval()

# Prepare the embedding model
embed_model_id = "sentence-transformers/all-MiniLM-L6-v2"
embed_tokenizer = AutoTokenizer.from_pretrained(embed_model_id)
embed_model = AutoModel.from_pretrained(embed_model_id).to('cuda')
_ = embed_model.eval()


**`generate_response()`**

Passes input text through a causal language model to generate a text completion using specified decoding parameters.

- prompt_text: The input string containing the formatted context and query.
- temperature: A float controlling the randomness of the model's token selection.
- top_p: A float controlling nucleus sampling, restricting selection to the smallest set of tokens whose cumulative probability exceeds this threshold.
- do_sample: A boolean determining whether to use probabilistic sampling or greedy decoding.
- max_new_tokens: An integer specifying the maximum number of new tokens the model is allowed to generate.

In [ ]:
def generate_response(prompt_text, temperature=1.0, top_p=1.0, do_sample=True, max_new_tokens=256):
    """
    Utilizes the optimized generate method to test decoding parameters.
    """
    inputs = tokenizer(prompt_text, return_tensors="pt").to('cuda')
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id
        )
        
    # Isolate the generated response from the input prompt
    input_length = inputs.input_ids.shape[1]
    response_ids = output_ids[0][input_length:]
    return tokenizer.decode(response_ids, skip_special_tokens=True)

**`get_text_embedding()`**

Computes a dense vector representation of input text using a specified encoder model and tokenizer, utilizing mean pooling with attention mask weighting.

- `text`: The input string to be converted into an embedding.
- `model`: The loaded encoder model.
- `tokenizer`: The associated tokenizer for the encoder model.

In [ ]:
def get_text_embedding(text, model=embed_model, tokenizer=embed_tokenizer):
    """
    Converts a text string into a dense vector using an encoder model, 
    accounting for attention masks during mean pooling.
    """
    # Tokenize with padding and truncation enabled
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512).to('cuda')
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    token_embeddings = outputs.last_hidden_state
    attention_mask = inputs.attention_mask
    
    # Expand the attention mask to match the embedding dimensions
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    
    # Multiply embeddings by the mask to zero-out padding tokens, then sum
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    
    # Calculate the number of non-padding tokens to divide by
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    # Calculate the true mean
    mean_pooled = sum_embeddings / sum_mask
    
    # Apply L2 normalization
    normalized_embedding = torch.nn.functional.normalize(mean_pooled, p=2, dim=1)
    
    return normalized_embedding.cpu().squeeze().to(torch.float32).numpy()

**`parse_and_chunk_document()`**
Reads a text file and splits it into discrete, overlapping text chunks based on sentence boundaries and maximum word count limits.

- `file_path`: A string specifying the path to the target text file.
- `max_words`: An integer specifying the maximum cumulative word count permitted in a single chunk.
- `overlap_sentences`: An integer specifying the number of sentences from the end of the previous chunk to include at the beginning of the next chunk.

In [ ]:
def parse_and_chunk_document(file_path, max_words=150, overlap_sentences=1):
    """
    Reads a plain text file, splits it into sentences, and groups them into 
    overlapping chunks to preserve semantic boundaries.
    """
    if not os.path.exists(file_path):
        print(f"Error: The file {file_path} does not exist.")
        return []

    # Parse the document
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()

    # Normalize whitespace to ensure consistent regex behavior
    text = re.sub(r'\s+', ' ', text).strip()

    # Split into sentences using a positive lookbehind for punctuation
    # This separates the string at spaces immediately following a period, question mark, or exclamation point
    sentences = re.split(r'(?<=[.!?])\s+', text)
    
    # Filter out any empty strings resulting from the split
    sentences = [s.strip() for s in sentences if s.strip()]

    chunks = []
    i = 0
    
    while i < len(sentences):
        current_chunk = []
        current_word_count = 0
        start_i = i
        
        # Accumulate sentences until the max_words limit is reached
        while i < len(sentences):
            sentence = sentences[i]
            word_count = len(sentence.split())
            
            # Always add the first sentence to avoid infinite loops on overly long sentences
            if not current_chunk or (current_word_count + word_count <= max_words):
                current_chunk.append(sentence)
                current_word_count += word_count
                i += 1
            else:
                break
                
        # Rejoin the grouped sentences into a single text chunk
        chunks.append(" ".join(current_chunk))
        
        # Apply overlap by stepping the index backward, ensuring forward progress is maintained
        if i < len(sentences):
            advance_step = max(1, (i - start_i) - overlap_sentences)
            i = start_i + advance_step

    return chunks

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; color: #856404;">
  <strong>🟡 AI Policy: YELLOW</strong> <br>
  Generative AI is allowed, with limitations.
</div>

### Tasks to understand predefined functions

Accomplish these tasks below to experiment with these functions.

**Tasks for generate_response**

1. Define a deterministic prompt string, such as `prompt = "The capital of the United States is"`
2. Execute the function with `prompt_text=prompt` and `do_sample=False`. Print the resulting string.
3. Execute the function a second time using the exact same prompt, but set `do_sample=True`. Run it a couple times to see the different results.

**Tasks for get_text_embedding**

1. Define a short text string, such as `test_string = "This is a test sentence."`
2. Pass this string into the function alongside `embed_model` and `embed_tokenizer`. Assign the return value to a variable named `vector`.
3. Print `type(vector)` and `vector.shape` to verify the function returns a 1D NumPy array with a length matching the embedding model's hidden size (384 for MiniLM-L6-v2).

**Tasks for parse_and_chunk_document**

1. Create a text file named `test_doc.txt` containing exactly five distinct sentences.
2. Execute the function with `file_path="test_doc.txt"`, `max_words=10`, and `overlap_sentences=1`.
3. Print the `len()` of the returned list to observe the total number of chunks created.
4. Print the string contents of index `0` and index `1` from the returned list to visually confirm the sentence overlap mechanism.

<div style="background-color: #fff3cd; border-left: 6px solid #ffc107; padding: 15px; color: #856404;">
  <strong>🟡 AI Policy: YELLOW</strong> <br>
  Generative AI is allowed, with limitations.
</div>

## Step 3: Implement RAG

To get an informed, less-hallucinated response from our LLM, we need to create a useful prompt which encodes system instructions, the user prompt, and information context from our document. The steps to do this are:

- Prepare the document
  - Break the document into chunks
  - Embed each chunk
  - Create a matrix of embeddings, which is `num_chunks x embedding_dimensions`
- Find the text in the document most relevant to the user's question
  - Embed the prompt
  - Find the chunks with embeddings most similar to the prompt embedding
  - Concatenate the text of these most-relevant chunks to make informational context
- Construct a prompt containing system instructions, user prompt, and informational context
- Ask the generator to complete the prompt.

For "Prepare the document," you already have functions to break up the document and embed a single chunk. Write a function `precompute_embeddings` which takes in a list of textual chunks and returns a matrix of embeddings. Save that matrix and the list of chunks to disk so you don't have to rebuild them every time.

For "Find the text in the document most relevant to the user's question" you already have a function to embed the prompt. Use `sklearn.neighbors.NearestNeighbors` with a metric of `cosine` to find the three chunks most relevant to the user's prompt. Concatenate that text together into a single string.

For "Construct a prompt containing system instructions, user prompt, and informational context," write a function which takes in those three things, and creates a single string with this format:
```
<|im_start|>system
your system instructions here<|im_end|>
<|im_start|>user
Context: your informational context here

Question: your user's prompt here<|im_end|
<|im_start|assistant
```

Test that function, and make sure the prompt injection worked correctly.

Feed that prompt into the causal model and let it complete the string!

<div style="background-color: #d4edda; border-left: 6px solid #28a745; padding: 15px; color: #155724;">
  <strong>🟢 AI Policy: GREEN</strong> <br>
  Generative AI is allowed/encouraged for this section.
</div>

## Part 4: Experiment

Here are some things you can play with:

- Get some new, longer documents to work with. Either edit your chunker to work on a list of filenames, or append all the text from multiple files together into one file. Dream big!
- Try some different models of different sizes. Gemini can help you identify models you can download.
- Play with some system messages. What types of behavior can you create?
- Add past interactions into the "Context" so it can carry out an extended conversation with knowledge of previous interactions.
- Make a clean Python program that makes it easy for someone to interact with it without being aware of all the behind-the-scenes work. Questions in, answers out!